# Análisis descriptivo de las lecturas

Describe lo que midió un nodo de GreenSense IoT: si los datos están completos, qué valores tomó cada variable, cómo cambian a lo largo del día y cuánto tiempo pasaron dentro del rango adecuado del cultivo.

**Datos:** el CSV que exporta la app (Historial, Lecturas CSV), guardado en `analisis/datos/`. Se usa el más reciente, o el que indique la variable de entorno `GS_CSV`. Todo el texto de resultados se calcula a partir de ese archivo.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

import comun

SERIE = "#2a78d6"
FRANJA = "#f0efec"
TINTA = "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.edgecolor": "#c9c8c3", "axes.labelcolor": TINTA,
    "xtick.color": TINTA, "ytick.color": TINTA, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e1",
    "lines.linewidth": 1.5,
})

ruta = comun.ruta_de_datos()
datos = comun.cargar(ruta)
print(f"Archivo: {ruta.name}")
print(f"Nodo: {', '.join(datos['nodo'].unique())}")

## 1. Calidad de los datos

Antes de interpretar nada: cuántas lecturas llegaron de las esperadas, a una por minuto, y dónde faltan.

In [ ]:
cal = comun.calidad(datos)
display(Markdown(
    f"- **Periodo:** {cal.primera:%d/%m/%Y %H:%M} a {cal.ultima:%d/%m/%Y %H:%M} ({cal.horas:.1f} horas, {cal.horas / 24:.1f} días)\n"
    f"- **Lecturas:** {cal.lecturas:,} de {cal.esperadas:,} esperadas\n"
    f"- **Perdidas:** {cal.perdidas:,} ({cal.porcentaje_perdidas:.2f} %); la meta de la propuesta es menos de 5 %\n"
    f"- **Huecos de más de 3 minutos:** {len(cal.huecos)}"
))
if cal.horas < 7 * 24:
    display(Markdown(
        f"**Limitación:** la propuesta pide 7 días continuos y aquí hay {cal.horas / 24:.1f}. "
        "Los resultados describen solo este periodo."
    ))
if not cal.huecos.empty:
    display(cal.huecos)

## 2. Estadísticas de cada variable

In [ ]:
comun.estadisticas(datos)

## 3. Las lecturas en el tiempo

La franja gris es el rango adecuado del cultivo. Los cortes en la línea son lecturas que faltan.

In [ ]:
presentes = [v for v in comun.VARIABLES if datos[v].notna().any()]
fig, ejes = plt.subplots(len(presentes), 1, figsize=(10, 2.5 * len(presentes)), sharex=True, squeeze=False)
rejilla = datos[presentes].resample("5min").median()
for eje, variable in zip(ejes[:, 0], presentes):
    eje.axhspan(*comun.RANGOS[variable], color=FRANJA, zorder=0)
    eje.plot(rejilla.index, rejilla[variable], color=SERIE)
    eje.set_ylabel(comun.NOMBRES[variable])
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 4. Ciclo diario

Promedio por hora del día; la banda va del percentil 10 al 90. Con pocos días, cada hora se apoya en pocas repeticiones.

In [ ]:
fig, ejes = plt.subplots(1, len(presentes), figsize=(4 * len(presentes), 3), squeeze=False)
frases = []
for eje, variable in zip(ejes[0], presentes):
    ciclo = comun.ciclo_diario(datos, variable)
    eje.fill_between(ciclo.index, ciclo["p10"], ciclo["p90"], color=SERIE, alpha=0.18, linewidth=0)
    eje.plot(ciclo.index, ciclo["promedio"], color=SERIE)
    eje.set_title(comun.NOMBRES[variable], fontsize=10, color=TINTA)
    eje.set_xlabel("Hora del día")
    eje.set_xticks(range(0, 24, 6))
    frases.append(
        f"- **{comun.NOMBRES[variable]}:** máximo promedio a las {ciclo['promedio'].idxmax()}:00 "
        f"({ciclo['promedio'].max():.1f}) y mínimo a las {ciclo['promedio'].idxmin()}:00 "
        f"({ciclo['promedio'].min():.1f})."
    )
fig.tight_layout()
plt.show()
display(Markdown("\n".join(frases)))

## 5. Tiempo dentro del rango adecuado

Porcentaje de las lecturas válidas. Los rangos son los del cultivo predeterminado; si el invernadero usa otro, hay que cambiar `comun.RANGOS`.

In [ ]:
comun.tiempo_en_rango(datos)

## 6. Relación entre variables

Correlación de Pearson, con las lecturas agrupadas cada 5 minutos. Una correlación no demuestra una causa: por ejemplo, la temperatura y la humedad del aire siguen las dos al ciclo del día.

In [ ]:
relacion = rejilla[[v for v in presentes if comun.tiene_variacion(datos, v)]].corr().round(2)
relacion

## 7. Lectura de los resultados

Texto generado a partir de los datos. **Antes de entregar, conviene completarlo con lo que se observó en el invernadero**, como la hora de cada riego.

In [ ]:
rangos = comun.tiempo_en_rango(datos)
lineas = [
    f"- Se analizaron {cal.lecturas:,} lecturas de {cal.horas / 24:.1f} días, con {cal.porcentaje_perdidas:.2f} % de pérdida "
    f"({'cumple' if cal.porcentaje_perdidas < 5 else 'no cumple'} la meta de menos de 5 %)."
]
for nombre, fila in rangos.iterrows():
    lineas.append(f"- {nombre}: {fila['dentro']:.1f} % del tiempo dentro del rango, {fila['debajo']:.1f} % debajo y {fila['arriba']:.1f} % arriba.")
if "humedad_sustrato" in presentes and not comun.tiene_variacion(datos, "humedad_sustrato"):
    lineas.append("- **Atención:** el sustrato casi no varió. Hay que revisar el sensor y su calibración antes de sacar conclusiones.")
display(Markdown("\n".join(lineas)))